# 3. LightGBM Baseline Training & Evaluation

Train baseline LightGBM model and perform xAI analysis.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.pipeline import DemandForecastPipeline
from src.xai.feature_importance import FeatureImportanceAnalyzer
from src.xai.shap_explainer import SHAPExplainer
from src.xai.visualization import plot_feature_importance, plot_predictions_vs_actual

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Run Full Pipeline

In [ ]:
# Configure pipeline
raw_path = './data/raw'
store_filter = 'CA_1'
split_date = '2016-04-24'

print(f"Starting demand forecast pipeline for {store_filter}...\n")

pipeline = DemandForecastPipeline(raw_path, split_date, store_filter)

# Run full pipeline
metrics, results = pipeline.run_full_pipeline(
    optimize_memory=True,
    model_name='lightgbm',
    num_rounds=500,
    early_stopping=50
)

print("\n" + "="*60)
print("PIPELINE COMPLETE - EVALUATION METRICS")
print("="*60)

## Model Evaluation

In [ ]:
# Display metrics
print(f"\nTest Set Performance:")
for metric, value in metrics.items():
    print(f"  {metric.upper():6s}: {value:.4f}")

# Show sample predictions
print(f"\nSample Predictions (first 10):")
display_results = results.head(10)[['id', 'date', 'actual_sales', 'pred']].copy()
print(display_results.to_string(index=False))

## Predictions vs Actual

In [ ]:
# Filter to rows with actual sales
valid_results = results[results['actual_sales'].notna()].copy()

plot_predictions_vs_actual(
    valid_results['actual_sales'].values,
    valid_results['pred'].values
)

print(f"\nPredictions Summary:")
print(f"  Actual sales - Mean: {valid_results['actual_sales'].mean():.2f}, Std: {valid_results['actual_sales'].std():.2f}")
print(f"  Predictions  - Mean: {valid_results['pred'].mean():.2f}, Std: {valid_results['pred'].std():.2f}")

## Feature Importance Analysis

In [ ]:
# Get model-based feature importance
gain_importance = pipeline.get_feature_importance(top_k=20)

print("\nTop 15 Features (Gain Importance):")
print(gain_importance.head(15).to_string(index=False))

# Visualize
plot_feature_importance(gain_importance, top_k=15, title="Top 15 Most Important Features")

## xAI: Permutation Importance

In [ ]:
# Get permutation-based importance
analyzer = FeatureImportanceAnalyzer(pipeline.model)

# Use subset for speed
test_subset = pipeline.X_test.head(100)
test_y_subset = pipeline.test_df['sales'].head(100)

perm_importance = analyzer.get_permutation_importance(
    test_subset,
    test_y_subset,
    n_repeats=5,
    top_k=15
)

print("\nTop 15 Features (Permutation Importance):")
print(perm_importance.to_string(index=False))

## xAI: SHAP Explainability

In [ ]:
# Initialize SHAP explainer
print("Initializing SHAP explainer...")
shap_explainer = SHAPExplainer(pipeline.model)
shap_explainer.fit(pipeline.X_train.head(100))  # Use subset for speed

# Get SHAP-based feature importance
shap_importance = shap_explainer.get_feature_importance_from_shap(
    pipeline.X_test.head(50),
    top_k=15
)

print("\nTop 15 Features (SHAP Importance):")
print(shap_importance.to_string(index=False))

## Compare Importance Methods

In [ ]:
# Compare methods on smaller subset
test_small = pipeline.X_test.head(50)
test_y_small = pipeline.test_df['sales'].head(50)

comparison = analyzer.compare_importance_methods(
    test_small,
    test_y_small,
    n_repeats=3,
    top_k=10
)

print("\nFeature Importance Comparison (Top 10):")
print(comparison[['feature', 'gain_normalized', 'perm_normalized']].to_string(index=False))

# Visualize comparison
from src.xai.visualization import plot_model_vs_permutation_importance
plot_model_vs_permutation_importance(comparison)

## Error Analysis

In [ ]:
# Calculate residuals
valid_results['residual'] = valid_results['actual_sales'] - valid_results['pred']
valid_results['abs_error'] = np.abs(valid_results['residual'])
valid_results['pct_error'] = np.abs(valid_results['residual']) / (valid_results['actual_sales'] + 1) * 100

print("\nError Statistics:")
print(f"  Mean Absolute Error: {valid_results['abs_error'].mean():.4f}")
print(f"  Median Absolute Error: {valid_results['abs_error'].median():.4f}")
print(f"  Max Absolute Error: {valid_results['abs_error'].max():.4f}")
print(f"  Mean Percentage Error: {valid_results['pct_error'].mean():.2f}%")

# Visualize error distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(valid_results['residual'], bins=50, edgecolor='black')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].grid(True, alpha=0.3)

axes[1].hist(valid_results['abs_error'], bins=50, edgecolor='black')
axes[1].set_xlabel('Absolute Error')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Absolute Error Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Results

In [ ]:
# Save model and results
from src.utils.helpers import save_pickle

output_dir = Path('./models/artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

# Save model
model_path = output_dir / f'lightgbm_{store_filter}.pkl'
save_pickle(pipeline.model.get_model(), model_path)

# Save predictions
results_path = output_dir / f'predictions_{store_filter}.parquet'
results.to_parquet(results_path)

# Save feature importance
importance_path = output_dir / f'feature_importance_{store_filter}.parquet'
gain_importance.to_parquet(importance_path)

print(f"✓ Model saved to {model_path}")
print(f"✓ Predictions saved to {results_path}")
print(f"✓ Feature importance saved to {importance_path}")